# Simple ANN Classifier — TinyML Anomaly Detector

Kiến trúc mạng phân loại nhị phân trực tiếp:

```
Input(2)  →  Dense(8, ReLU)  →  Dense(1, Sigmoid)
```

| Layer | Units | Activation | Vai trò |
|-------|-------|------------|---------|
| Input | 2 | — | [Temperature, Humidity] đã chuẩn hóa Z-score |
| Hidden | **8** | **ReLU** | Học đặc trưng |
| Output | **1** | **Sigmoid** | Xác suất là Anomaly (0 → 1) |

**Cách phát hiện bất thường:** `output > 0.5` → **ANOMALY**, `output ≤ 0.5` → Normal.

**Ngưỡng bình thường:**
- Temp: 25.0 – 30.0 °C  
- Humi: 40.0 – 60.0 %

---
> ⚠️ **Tại sao không dùng PyTorch?**  
> `TensorFlowLite_ESP32` chỉ đọc định dạng **`.tflite` flatbuffer**.  
> → **Phải dùng TensorFlow/Keras** để xuất `.tflite` trực tiếp.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow", "scikit-learn", "pandas"], check=True)

CompletedProcess(args=['d:\\Programs\\Python3.11.9\\python.exe', '-m', 'pip', 'install', 'tensorflow', 'scikit-learn', 'pandas'], returncode=0)

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler

# === THRESHOLD CONSTANTS (khớp với global.h) ===
TEMP_NORMAL_LOW  = 25.0
TEMP_NORMAL_HIGH = 30.0
HUMI_NORMAL_LOW  = 40.0
HUMI_NORMAL_HIGH = 60.0

# --- 1. SINH DỮ LIỆU GIẢ LẬP ---
np.random.seed(42)
num_normal  = 1600
num_anomaly = 400

normal_temp = np.random.uniform(TEMP_NORMAL_LOW,  TEMP_NORMAL_HIGH, num_normal)
normal_humi = np.random.uniform(HUMI_NORMAL_LOW,  HUMI_NORMAL_HIGH, num_normal)

anom_temp = np.concatenate([
    np.random.uniform(TEMP_NORMAL_HIGH + 5, 45.0, 100),  # Quá nóng
    np.random.uniform(10.0, TEMP_NORMAL_LOW  - 5, 100),  # Quá lạnh
    np.random.uniform(TEMP_NORMAL_LOW, TEMP_NORMAL_HIGH, 100),
    np.random.uniform(TEMP_NORMAL_LOW, TEMP_NORMAL_HIGH, 100),
])
anom_humi = np.concatenate([
    np.random.uniform(HUMI_NORMAL_LOW, HUMI_NORMAL_HIGH, 200),
    np.random.uniform(HUMI_NORMAL_HIGH + 10, 95.0, 100),  # Quá ẩm
    np.random.uniform(5.0, HUMI_NORMAL_LOW  - 10, 100),  # Quá khô
])

df = pd.DataFrame({
    'Temperature': np.concatenate([normal_temp, anom_temp]),
    'Humidity':    np.concatenate([normal_humi, anom_humi]),
    'Anomaly':     np.concatenate([np.zeros(num_normal), np.ones(num_anomaly)])
}).sample(frac=1).reset_index(drop=True)

df.to_csv('custom_iot_dataset.csv', index=False)
print(f"Dataset: {len(df)} mẫu ({num_normal} bình thường, {num_anomaly} bất thường)")

# --- 2. CHUẨN HÓA (fit trên toàn bộ dữ liệu train) ---
X = df[['Temperature', 'Humidity']].values
y = df['Anomaly'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"SCALER_MEAN  = [{scaler.mean_[0]:.4f}, {scaler.mean_[1]:.4f}]")
print(f"SCALER_SCALE = [{scaler.scale_[0]:.4f}, {scaler.scale_[1]:.4f}]")

Dataset: 2000 mẫu (1600 bình thường, 400 bất thường)
SCALER_MEAN  = [27.4874, 49.9192]
SCALER_SCALE = [4.2157, 12.0575]


In [3]:
# --- 3. XÂY DỰNG ANN CLASSIFIER ---
#
#   Input(2) → Dense(8, ReLU) → Dense(1, Sigmoid)
#
# Phân loại nhị phân trực tiếp: output > 0.5 → ANOMALY

model = models.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
], name="anomaly_classifier")

model.summary()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_scaled, y,
    epochs=100,
    batch_size=16,
    validation_split=0.1,
    verbose=0
)
print(f"Final train loss:     {history.history['loss'][-1]:.6f}")
print(f"Final train accuracy: {history.history['accuracy'][-1]:.4f}")

ANOMALY_THRESHOLD = 0.5

print(f"\n=== SAO CHÉP GIÁ TRỊ NÀY VÀO src/tinyml/tinyml.cpp ===")
print(f'const float SCALER_MEAN[]      = {{{scaler.mean_[0]:.4f}f, {scaler.mean_[1]:.4f}f}};')
print(f'const float SCALER_SCALE[]     = {{{scaler.scale_[0]:.4f}f, {scaler.scale_[1]:.4f}f}};')
print(f'const float ANOMALY_THRESHOLD  = {ANOMALY_THRESHOLD}f;')

Model: "anomaly_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33 (132.00 B)

 Trainable params: 33 (132.00 B)

 Non-trainable params: 0 (0.00 B)

Final train loss:     0.022765
Final train accuracy: 0.9944

=== SAO CHÉP GIÁ TRỊ NÀY VÀO src/tinyml/tinyml.cpp ===
const float SCALER_MEAN[]      = {27.4874f, 49.9192f};
const float SCALER_SCALE[]     = {4.2157f, 12.0575f};
const float ANOMALY_THRESHOLD  = 0.5f;


In [4]:
# --- 4. KIỂM TRA VỚI CÁC NGƯỠNG BIÊN ---
print(f"Threshold: {ANOMALY_THRESHOLD}\n")

test_cases = pd.DataFrame([
    [TEMP_NORMAL_LOW  + 2.5, HUMI_NORMAL_LOW  + 10],   # Giữa vùng bình thường → Normal
    [TEMP_NORMAL_HIGH - 0.1, HUMI_NORMAL_HIGH - 0.1],   # Sát biên trên         → Normal
    [TEMP_NORMAL_HIGH + 5.5, HUMI_NORMAL_LOW  + 10],    # Quá nóng              → ANOMALY
    [TEMP_NORMAL_LOW  - 5.0, HUMI_NORMAL_LOW  + 10],    # Quá lạnh              → ANOMALY
    [TEMP_NORMAL_LOW  + 2.5, HUMI_NORMAL_HIGH + 10],    # Quá ẩm               → ANOMALY
    [TEMP_NORMAL_LOW  + 2.5, HUMI_NORMAL_LOW  - 15],    # Quá khô              → ANOMALY
], columns=['Temperature', 'Humidity'])

test_scaled = scaler.transform(test_cases)
scores = model.predict(test_scaled, verbose=0).flatten()

labels = ["Normal (giữa)", "Normal (biên)", "ANOMALY (nóng)",
          "ANOMALY (lạnh)", "ANOMALY (ẩm)", "ANOMALY (khô)"]

print(f"{'Input':>28} {'Score':>8}  {'Kết quả':>15}  {'Mong đợi'}")
print("-" * 75)
for i in range(len(test_cases)):
    result   = "ANOMALY" if scores[i] > ANOMALY_THRESHOLD else "Normal"
    expected = labels[i]
    ok = "✓" if (result == "Normal") == ("Normal" in expected) else "✗"
    print(f"T={test_cases.values[i][0]:4.1f} H={test_cases.values[i][1]:4.1f}  "
          f"{scores[i]:>8.4f}  {result:>15}  {ok} {expected}")

Threshold: 0.5

                       Input    Score          Kết quả  Mong đợi
---------------------------------------------------------------------------
T=27.5 H=50.0    0.0001           Normal  ✓ Normal (giữa)
T=29.9 H=59.9    0.0001           Normal  ✓ Normal (biên)
T=35.5 H=50.0    0.9620          ANOMALY  ✓ ANOMALY (nóng)
T=20.0 H=50.0    0.9319          ANOMALY  ✓ ANOMALY (lạnh)
T=27.5 H=70.0    0.7363          ANOMALY  ✓ ANOMALY (ẩm)
T=27.5 H=25.0    0.9864          ANOMALY  ✓ ANOMALY (khô)


d:\Programs\Python3.11.9\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [5]:
# --- 5. XUẤT RA FILE C++ và TỰ ĐỘNG REPLACE include/tinyml/dht_model_v2.h ---
import pathlib, shutil

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

def hex_to_c_array(hex_data, var_name):
    c_str = f"const unsigned char {var_name}[] = {{\n  "
    for i, byte in enumerate(hex_data):
        c_str += f"0x{byte:02x}, "
        if (i + 1) % 12 == 0:
            c_str += "\n  "
    c_str = c_str.strip(", \n") + "\n};\n"
    c_str += f"const unsigned int {var_name}_len = {len(hex_data)};"
    return c_str

# Tìm project root (thư mục chứa platformio.ini)
root = pathlib.Path().resolve()
for _ in range(6):
    if (root / "platformio.ini").exists():
        break
    root = root.parent

dest_file = root / "include" / "tinyml" / "dht_model_v2.h"
dest_file.parent.mkdir(parents=True, exist_ok=True)

with open(dest_file, "w", encoding="utf-8") as f:
    f.write(hex_to_c_array(tflite_model, "dht_anomaly_model_tflite"))

print(f"Model size : {len(tflite_model)} bytes")
print(f"Replaced   : {dest_file}")
print()
print("Cap nhat src/tinyml/tinyml.cpp:")
print(f'  const float SCALER_MEAN[]     = {{{scaler.mean_[0]:.4f}f, {scaler.mean_[1]:.4f}f}};')
print(f'  const float SCALER_SCALE[]    = {{{scaler.scale_[0]:.4f}f, {scaler.scale_[1]:.4f}f}};')
print(f'  const float ANOMALY_THRESHOLD = {ANOMALY_THRESHOLD}f;')

INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpssrgdptp\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpssrgdptp\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpssrgdptp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 2), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1448803320848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1448231700624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1448231699472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1448231698512: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model size : 1928 bytes
Replaced   : D:\Documents\252\iot\iot_assignment\YoloUNO_PlatformIO-RTOS_Project\include\tinyml\dht_model_v2.h

Cap nhat src/tinyml/tinyml.cpp:
  const float SCALER_MEAN[]     = {27.4874f, 49.9192f};
  const float SCALER_SCALE[]    = {4.2157f, 12.0575f};
  const float ANOMALY_THRESHOLD = 0.5f;


In [9]:
# --- 6. TEST ĐỘC LẬP — Load trực tiếp từ include/tinyml/dht_model_v2.h ---
# Cell này chạy độc lập, không cần chạy các cell trên.
# Nếu vừa train lại, cập nhật SCALER_MEAN / SCALER_SCALE từ output cell 3.
import re, pathlib, numpy as np
import tensorflow as tf

# -- Tìm project root --
root = pathlib.Path().resolve()
for _ in range(6):
    if (root / "platformio.ini").exists():
        break
    root = root.parent

h_file = root / "include" / "tinyml" / "dht_model_v2.h"
content = h_file.read_text(encoding="utf-8")
model_bytes = bytes(int(b, 16) for b in re.findall(r'0x([0-9a-fA-F]{2})', content))

# -- Load TFLite interpreter --
interp = tf.lite.Interpreter(model_content=model_bytes)
interp.allocate_tensors()
inp_idx = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']

# -- Scaler constants (đồng bộ với output cell 3) --
SCALER_MEAN       = [27.4874, 49.9192]
SCALER_SCALE      = [4.2157,  12.0575]
ANOMALY_THRESHOLD = 0.5

print(f"Loaded {len(model_bytes)} bytes from {h_file.name}\n")

# -- Nhập input --
temp = 15
humi = 40

inp = np.array([[(temp - SCALER_MEAN[0]) / SCALER_SCALE[0],
                  (humi - SCALER_MEAN[1]) / SCALER_SCALE[1]]], dtype=np.float32)
interp.set_tensor(inp_idx, inp)
interp.invoke()
score = float(interp.get_tensor(out_idx)[0][0])

result = "ANOMALY !" if score > ANOMALY_THRESHOLD else "Normal"
print(f"\nT={temp:.1f}C  H={humi:.1f}%  ->  Score: {score:.4f}  ->  {result}")

Loaded 1928 bytes from dht_model_v2.h


T=15.0C  H=40.0%  ->  Score: 0.9965  ->  ANOMALY !
